# EDA on Latest R1 Features Dataset

This notebook loads the latest `features_YYYYMMDD_HHMMSS.csv` file from `data/processed/` and performs a structured EDA on the processed R1 feature table.

The emphasis is on:

- what one row in the final dataset actually represents
- missing values and coverage
- delay distributions and temporal patterns
- `direction` coverage and why some directions are still missing
- the new timetable-derived `stop_sequence`

Important context: the `features` dataset is **not** raw minute-level timetable history. Each row is already an aggregated `train_id + station_id` record built from the raw timetable observations.

In [ ]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

In [ ]:
# Resolve project paths relative to this notebook.
PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TIMETABLE_DIR = PROJECT_DIR / "EDA" / "official_timetables"


def normalize_station_name(name):
    if pd.isna(name):
        return np.nan

    normalized = unicodedata.normalize("NFKD", str(name))
    normalized = "".join(ch for ch in normalized if not unicodedata.combining(ch))
    normalized = normalized.lower()
    normalized = normalized.replace("|", " ")
    normalized = normalized.replace("-", " ")
    normalized = normalized.replace(".", " ")
    normalized = re.sub(r"\bst\b", "sant", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()

    aliases = {
        "barcelona el clot": "barcelona el clot arago",
        "barcelona placa de catalunya": "barcelona placa catalunya",
    }
    return aliases.get(normalized, normalized)


def latest_features_file(processed_dir: Path) -> Path:
    candidates = sorted(
        path
        for path in processed_dir.glob("features_*.csv")
        if re.fullmatch(r"features_\d{8}_\d{6}\.csv", path.name)
    )
    if not candidates:
        raise FileNotFoundError("No timestamped features CSV found in data/processed/.")
    return candidates[-1]


def load_r1_reference_orders(timetable_dir: Path):
    dir1 = pd.read_csv(timetable_dir / "R1_direction1_schedules.csv", nrows=0)
    dir2 = pd.read_csv(timetable_dir / "R1_direction2_schedules.csv", nrows=0)

    stations_dir1 = [c for c in dir1.columns if c != "Day_Type"]
    stations_dir2 = [c for c in dir2.columns if c != "Day_Type"]

    order_dir1 = {normalize_station_name(s): idx for idx, s in enumerate(stations_dir1)}
    order_dir2 = {normalize_station_name(s): idx for idx, s in enumerate(stations_dir2)}

    return stations_dir1, stations_dir2, order_dir1, order_dir2

In [ ]:
# Load the latest processed features file.
features_path = latest_features_file(PROCESSED_DIR)
df = pd.read_csv(features_path)

datetime_cols = [
    "first_planned_arrival",
    "last_planned_arrival",
    "first_planned_departure",
    "last_planned_departure",
    "last_actual_arrival",
    "last_actual_departure",
    "planned_arrival_dt",
    "actual_arrival_dt",
    "hour_trunc",
]
for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "service_date" in df.columns:
    df["service_date"] = pd.to_datetime(df["service_date"], errors="coerce")

stations_dir1, stations_dir2, order_dir1, order_dir2 = load_r1_reference_orders(TIMETABLE_DIR)
reference_station_set = set(order_dir1) & set(order_dir2)
terminal_station_names = {stations_dir1[0], stations_dir1[-1]}
terminal_station_set = {normalize_station_name(s) for s in terminal_station_names}

print(f"Loaded features file: {features_path.name}")
print(f"Rows: {len(df):,}")
print(f"Unique trains: {df['train_id'].nunique():,}")
print(f"Unique stations: {df['station_name'].nunique():,}")
print(f"Date range: {df['service_date'].min().date()} -> {df['service_date'].max().date()}")

## 1. What Does One Row Represent?

This is the most important point for interpreting the plots correctly.

A row in `features_*.csv` is **not** one raw observation and it is **not** one minute of service.

Instead, the build pipeline aggregates the raw timetable history by:

- `train_id`
- `station_id`

and keeps summary fields such as:

- first / last planned time
- last actual time
- derived delay features
- inferred direction
- timetable-derived stop sequence

So when we count rows by date, we are counting **aggregated train-station records**, not minute-level API polls.

In [ ]:
display(df.head())
display(df.describe(include="all").transpose().head(30))

## 2. Missingness Overview

This section shows which columns still have null values and how important that missingness is relative to dataset size.

In [ ]:
missing_summary = (
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": 100 * df.isna().mean(),
    })
    .sort_values("missing_count", ascending=False)
)
display(missing_summary)

plt.figure(figsize=(10, 6))
missing_summary.query("missing_count > 0").sort_values("missing_count").plot(
    kind="barh", y="missing_count", legend=False, color="#C44E52"
)
plt.title("Missing Values by Column")
plt.xlabel("Missing count")
plt.ylabel("")
plt.tight_layout()
plt.show()

### Train-level examples behind missing `direction`

The key question is not just *which rows* have missing direction, but whether those trains retain only one stop in the final aggregated dataset.

The reasoning is:

1. Filter the final features table to rows where `direction` is missing.
2. Group by `train_id` in the **final aggregated dataset**.
3. Count how many retained rows each train has.
4. Count how many unique stations each train has.
5. Inspect the full retained station list for sample unresolved trains.

If the grouped counts show `train_row_count = 1` and `unique_station_count = 1`, then that train contributes only one retained train-station row to the final dataset. In that case, direction cannot be inferred because there is no previous/next station inside the processed table to compare against.

In [ ]:
train_level_counts = (
    df.groupby("train_id")
    .agg(
        train_row_count=("station_id", "size"),
        unique_station_count=("station_id", "nunique"),
    )
    .reset_index()
)

missing_direction_trains = (
    df[df["direction"].isna()]
    [["train_id"]]
    .drop_duplicates()
    .merge(train_level_counts, on="train_id", how="left")
    .sort_values(["train_row_count", "unique_station_count", "train_id"])
)

print("Train-level counts for trains with missing direction:")
display(missing_direction_trains.head(20))

print("\nDistribution of retained row counts among trains with missing direction:")
display(
    missing_direction_trains["train_row_count"]
    .value_counts()
    .sort_index()
    .rename_axis("train_row_count")
    .reset_index(name="train_count")
)

print("\nDistribution of unique station counts among trains with missing direction:")
display(
    missing_direction_trains["unique_station_count"]
    .value_counts()
    .sort_index()
    .rename_axis("unique_station_count")
    .reset_index(name="train_count")
)

missing_direction_examples = (
    df[df["direction"].isna()]
    .merge(train_level_counts, on="train_id", how="left")
    [["train_id", "train_row_count", "unique_station_count", "station_id", "station_name", "planned_arrival_dt", "actual_arrival_dt", "stop_sequence"]]
    .sort_values(["train_row_count", "station_name", "train_id"])
)

print("Sample rows with missing direction:")
display(missing_direction_examples.head(20))

sample_train_ids = missing_direction_examples["train_id"].drop_duplicates().head(8).tolist()
train_station_lists = (
    df[df["train_id"].isin(sample_train_ids)]
    .merge(train_level_counts, on="train_id", how="left")
    [["train_id", "train_row_count", "unique_station_count", "station_id", "station_name", "planned_arrival_dt", "actual_arrival_dt", "direction", "stop_sequence"]]
    .sort_values(["train_id", "planned_arrival_dt", "station_id"])
)

print("\nAll retained station rows for a sample of missing-direction train IDs:")
display(train_station_lists)

direction_station_summary = (
    df[df["direction"].isna()]
    .groupby(["station_id", "station_name"], dropna=False)
    .agg(
        missing_direction_rows=("train_id", "size"),
        missing_direction_trains=("train_id", "nunique"),
    )
    .reset_index()
    .sort_values(["missing_direction_rows", "missing_direction_trains"], ascending=[False, False])
)
print("\nStation IDs and names associated with missing direction:")
display(direction_station_summary.head(20))

## 3. Filtered Dataset for Downstream EDA

From this point onward, the main EDA uses a filtered version of the dataset where rows with missing `direction` are removed.

This means the later delay, stop-sequence, and correlation plots are computed on the subset of rows where direction is available.

The missing-direction diagnostics remain useful as a separate analysis of the rows that were excluded.

In [ ]:
df_model = df[df["direction"].notna()].copy()

print(f"Original rows: {len(df):,}")
print(f"Rows after removing missing direction: {len(df_model):,}")
print(f"Dropped rows: {len(df) - len(df_model):,}")

## 4. Temporal Coverage and Volume Patterns

The plots below use `df_model`, so they show how many **aggregated train-station rows with non-missing direction** are present for each `service_date`.

They do **not** show raw minute-level timetable observations.

A spike means that, after grouping the raw history by `train_id + station_id`, that day still produced many final feature rows. A low value means the final dataset contains fewer train-station records for that day.

The first plot keeps the full scale so you can see spike days. The second plot removes the most extreme days from the y-axis focus so the non-peak part is easier to read.

In [ ]:
daily_counts = df_model.groupby("service_date").size().rename("rows").reset_index()
hour_counts = df_model.groupby("hour").size().rename("rows").reset_index()
peak_threshold = daily_counts["rows"].quantile(0.95)
daily_counts_zoom = daily_counts[daily_counts["rows"] <= peak_threshold].copy()

display(daily_counts.sort_values("rows", ascending=False).head(10))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

sns.lineplot(data=daily_counts, x="service_date", y="rows", marker="o", ax=axes[0], color="#4C72B0")
axes[0].set_title("Rows per Service Date (Full Scale)")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=daily_counts_zoom, x="service_date", y="rows", ax=axes[1], color="#64B5CD")
axes[1].set_title("Rows per Service Date (Zoomed, Peaks Removed)")
axes[1].tick_params(axis="x", rotation=90)

sns.barplot(data=hour_counts, x="hour", y="rows", ax=axes[2], color="#55A868")
axes[2].set_title("Rows by Planned Arrival Hour")
axes[2].set_xlabel("planned arrival hour")

plt.tight_layout()
plt.show()

## 5. Delay EDA

The target in this dataset is `target_delay`, which is currently equal to `delay_type_2`. The plots below are computed on the filtered dataset `df_model` and show how delay varies across day types, directions, and the busiest stations among rows with known direction.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df_model["target_delay"], bins=50, kde=True, ax=axes[0], color="#4C72B0")
axes[0].set_title("Target Delay Distribution")

sns.boxplot(data=df_model, x="day_type", y="target_delay", ax=axes[1], color="#DD8452")
axes[1].set_title("Target Delay by Day Type")
axes[1].tick_params(axis="x", rotation=20)

sns.boxplot(data=df_model, x="direction", y="target_delay", ax=axes[2], color="#55A868")
axes[2].set_title("Target Delay by Direction")

plt.tight_layout()
plt.show()

top_stations = df_model["station_name"].value_counts().head(10).index
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_model[df_model["station_name"].isin(top_stations)],
    x="station_name",
    y="target_delay",
    color="#8172B3",
)
plt.title("Target Delay by Top 10 Stations")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Timetable-Derived Stop Sequence

In the current build, `stop_sequence` is derived from the official R1 timetable order after direction inference.

That means:

- it no longer comes from the raw parquet `stop_sequence`
- it is deterministic once `direction` is known
- it remains null when `direction` is null, because the direction is needed to decide which station order to use

Because this section uses `df_model`, it focuses on the rows where direction is already known, so the remaining stop-sequence profile should be much cleaner.

In [ ]:
stop_sequence_summary = pd.DataFrame({
    "missing_count": [df_model["stop_sequence"].isna().sum()],
    "missing_pct": [100 * df_model["stop_sequence"].isna().mean()],
    "non_missing_unique_values": [df_model["stop_sequence"].dropna().nunique()],
})
display(stop_sequence_summary)

plt.figure(figsize=(8, 5))
sns.histplot(df_model["stop_sequence"].dropna(), bins=30, color="#4C72B0")
plt.title("Distribution of Timetable-Derived Stop Sequence")
plt.tight_layout()
plt.show()

## 7. Numeric Relationships

A correlation heatmap helps check whether the engineered delay variables and temporal features move together in plausible ways.

In [ ]:
numeric_cols = [
    "stop_sequence",
    "delay_type_1",
    "delay_type_2",
    "delay_masking_minutes",
    "target_delay",
    "hour",
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "prev_station_delay",
]
corr = df_model[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 8. Direction Coverage in the Filtered Dataset

After filtering, this section should confirm that the working dataset contains only resolved direction values.

In [ ]:
direction_summary = (
    df_model["direction"]
    .fillna("missing")
    .value_counts(dropna=False)
    .rename_axis("direction")
    .reset_index(name="rows")
)
direction_summary["pct"] = 100 * direction_summary["rows"] / len(df_model)
display(direction_summary)

plt.figure(figsize=(8, 4))
sns.barplot(data=direction_summary, x="direction", y="rows", palette="Set2")
plt.title("Direction Coverage")
plt.tight_layout()
plt.show()

## 9. Why Was Direction Missing in the Removed Rows?

The direction inference logic needs at least two matched R1 reference stations for a train. This section rebuilds that logic at train level and assigns an interpretable reason to each unresolved train.

This is the section that answers questions like:

- does the train have only one observed station in the final dataset?
- does it never expose two usable stations to infer direction?
- is the unresolved case concentrated at terminal stations or not?

In [ ]:
train_diag = df.copy()
train_diag["direction_missing"] = train_diag["direction"].isna()
train_diag["is_reference_station"] = train_diag["station_name_normalized"].isin(reference_station_set)
train_diag["is_terminal_station"] = train_diag["station_name_normalized"].isin(terminal_station_set)

train_summary = (
    train_diag.groupby("train_id")
    .agg(
        train_row_count=("station_id", "size"),
        unique_station_count=("station_name", "nunique"),
        unique_reference_station_count=("station_name_normalized", lambda s: s[s.isin(reference_station_set)].nunique()),
        any_missing_direction=("direction_missing", "any"),
        non_null_stop_sequence_count=("stop_sequence", lambda s: s.notna().sum()),
        first_station=("station_name", "first"),
        last_station=("station_name", "last"),
        terminal_station_observed=("is_terminal_station", "max"),
    )
    .reset_index()
)


def classify_reason(row):
    if not row["any_missing_direction"]:
        return "direction_inferred"
    if row["train_row_count"] == 1 and row["terminal_station_observed"]:
        return "single_station_observation_at_terminal"
    if row["train_row_count"] == 1:
        return "single_station_observation"
    if row["unique_reference_station_count"] < 2:
        return "fewer_than_two_reference_stations"
    return "ambiguous_after_reference_matching"


train_summary["missing_direction_reason"] = train_summary.apply(classify_reason, axis=1)
missing_train_summary = train_summary[train_summary["any_missing_direction"]].copy()
display(missing_train_summary.head())

In [ ]:
reason_summary = (
    missing_train_summary["missing_direction_reason"]
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="train_count")
)
display(reason_summary)

plt.figure(figsize=(10, 4))
sns.barplot(data=reason_summary, x="reason", y="train_count", color="#C44E52")
plt.title("Reasons for Missing Direction")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 10. Which Stations Are Behind the Missing Directions?

This station-level view combines row counts, train counts, and train-level reasons so you can see whether unresolved cases are concentrated at specific stops.

In [ ]:
missing_rows = df[df["direction"].isna()].copy()
missing_rows = missing_rows.merge(
    missing_train_summary[["train_id", "missing_direction_reason", "train_row_count", "unique_reference_station_count"]],
    on="train_id",
    how="left",
)

missing_station_summary = (
    missing_rows.groupby(["station_name", "missing_direction_reason"], dropna=False)
    .agg(
        row_count=("train_id", "size"),
        train_count=("train_id", "nunique"),
        avg_train_row_count=("train_row_count", "mean"),
        avg_reference_station_count=("unique_reference_station_count", "mean"),
    )
    .reset_index()
    .sort_values(["row_count", "train_count"], ascending=[False, False])
)
display(missing_station_summary)

plt.figure(figsize=(10, 5))
station_totals = missing_rows["station_name"].value_counts().rename_axis("station_name").reset_index(name="row_count")
sns.barplot(data=station_totals, x="row_count", y="station_name", color="#8172B3")
plt.title("Missing Direction Rows by Station")
plt.tight_layout()
plt.show()

## 11. Train-Level Examples to Inspect Manually

These tables expose the concrete `train_id`s behind unresolved directions. This is useful if you want to compare the processed result against the raw timetable history for a handful of trains.

In [ ]:
example_missing_trains = missing_train_summary.sort_values(
    ["missing_direction_reason", "train_row_count", "train_id"]
).copy()
display(example_missing_trains.head(25))

display(
    missing_rows[
        [
            "train_id",
            "station_name",
            "planned_arrival_dt",
            "actual_arrival_dt",
            "stop_sequence",
            "prev_station_delay",
            "missing_direction_reason",
        ]
    ].sort_values(["missing_direction_reason", "station_name", "train_id"]).head(50)
)

## 12. Additional EDA Cuts on the Filtered Dataset

These plots also use `df_model`, so they describe only the subset where direction is known.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sample_df = df_model.sample(min(len(df_model), 2500), random_state=42)
sns.scatterplot(data=sample_df, x="prev_station_delay", y="target_delay", alpha=0.35, ax=axes[0, 0], color="#4C72B0")
axes[0, 0].set_title("Target Delay vs Previous Station Delay")

sns.boxplot(data=df_model, x="is_weekend", y="target_delay", ax=axes[0, 1], color="#55A868")
axes[0, 1].set_title("Target Delay by Weekend Flag")

hour_delay = df_model.groupby("hour")["target_delay"].mean().reset_index()
sns.lineplot(data=hour_delay, x="hour", y="target_delay", marker="o", ax=axes[1, 0], color="#C44E52")
axes[1, 0].set_title("Average Target Delay by Hour")

station_delay = (
    df_model.groupby("station_name")
    .agg(rows=("train_id", "size"), avg_target_delay=("target_delay", "mean"))
    .query("rows >= 50")
    .sort_values("avg_target_delay", ascending=False)
    .head(10)
    .reset_index()
)
sns.barplot(data=station_delay, x="avg_target_delay", y="station_name", ax=axes[1, 1], color="#DD8452")
axes[1, 1].set_title("Top Stations by Average Target Delay (min 50 rows)")

plt.tight_layout()
plt.show()

## 13. Interpretation Notes

- After section 2, the main EDA uses `df_model`, which excludes rows with missing `direction`.
- The daily row-count plots operate on the final aggregated feature table, not the raw observation history.
- A missing `direction` generally means the train never exposed enough usable stations in the final dataset to infer travel order.
- A missing `stop_sequence` now mostly means the same thing, because the sequence is derived from the official timetable after direction inference.
- Terminal-station singletons are the best candidates for optional heuristics if you later decide to fill a small subset of unresolved directions.
- The train-level tables are the best starting point if you want to manually debug individual unresolved cases against the raw parquet history.